# CARISMA - Download and Extract Magnetometer Data

***

**Tutorial:** This tutorial explains how to extract CARISMA magnetometer and CANOPUS riometer data from the open data portal.  
**Mission and Instrument:** CARISMA (Canadian Array for Realtime Investigations of Magnetic Activity)  
**Astronomical Target:** Measure Earth's magnetic field to study space weather events like geomagnetic storms and substorms.  
**System Requirements:** Access to the internet.  
**Tutorial Level:** Intermediate

The CARISMA data can be found in both CSV and raw datasets on the CSA Open Data Portal. The raw data can be found [here](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma/) and the CSV data can be found [here](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma_csv/). Additionally, CARISMA data is also hosted by the University of Alberta at [carisma.ca](https://www.carisma.ca/).

# Part 1: Downloading CARISMA data

The CARISMA/CANOPUS data is hosted on the [CSA Open Data Portal](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub). Files can be downloaded by web scraping.

# 1.1 Web Scraping

### Directory Structure 

```
/users/OpenData_DonneesOPuvertes/pub/
|
|-- carisma_csv/                                    <-- CARISMA magnetometer data (csv)
|
|-- CANOPUS_CSV/                                    <-- CANOPUS riometer data (csv)
|   +-- old_canopus_riometer_format/                <-- Old riometer format
|
|-- carisma/                                        <-- RAW CARISMA files (.tar)
|
|-- CANOPUS/                                        <-- Raw CANOPUS files (.tar.gz)
```
Magnetometer stations: BACK, CONT, DAWS, ESKI, FCHU, FSIM, FSMI, GILL, GULL, ISLL, MCMU, MSTK, PINA, RABB, RANK, TALO


### Scraping the Directory Listing

The portal returns an HTML index page for each directory. We can scrape the `href` links to discover available years, stations, and files.

In [ ]:
# %pip install pandas matplotlib

In [ ]:
import os 
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from io import StringIO
from datetime import datetime, timedelta

# The CSA server will throw a self-signed certificate error, so we need to disable SSL verification
requests.packages.urllib3.disable_warnings()

BASE_URL = "https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/"
MAG_BASE = BASE_URL + "CARISMA/carisma_csv/mag/daily/"

In [ ]:
def list_directory(url):
    resp = requests.get(url, verify=False)
    
    links = re.findall(r'href="([^"]+)"', resp.text)

    # Filter parent-directory links, query strings
    names = []
    for link in links:
        name = link.strip('/').split('/')[-1]
        if name and name != ".." and "?" not in link and link != "../":
            # Skip links that point to parent path
            if not link.startswith("/"):
                names.append(name)
    return names    


# Browse the magnetometer data archive
print("Available years of magnetometer data:")
years = list_directory(MAG_BASE)
print(years)

# List stations for 2008
print("\nAvailable stations for 2008:")
stations = list_directory(MAG_BASE + "2008/")
print(stations)

# List first few files for station GILL
print("\nFirst 5 files for station GILL in 2008:")
files = list_directory(MAG_BASE + "2008/GILL/")
print(files[:5])


### Part 1: Download Magnetometer Data

#### 1.1 Download Programatically

In [ ]:
def download_mag_file(station, date_str, save_dir="data/mag"):
    year = date_str[:4]
    filename = f"{date_str}{station}.MAG.csv"
    url = f"{MAG_BASE}{year}/{station}/{filename}"

    os.makedirs(save_dir, exist_ok=True)
    local_path = os.path.join(save_dir, filename)

    try:
        resp = requests.get(url, verify=False)
        with open(local_path, 'wb') as f:
            f.write(resp.content)
        print(f"Downloaded {filename} to {local_path}")
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
        return None
    
    return local_path

# Download one day of data for GILL station (e.g., January 1, 2008)
download_mag_file("GILL", "20080101")

In [ ]:
def download_mag_batch(station, start_date, end_date, save_dir="data/mag"):
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")

    os.makedirs(save_dir, exist_ok=True)

    downloaded = []
    current = start

    while current <= end:
        date_str = current.strftime("%Y%m%d")
        year = current.strftime("%Y")
        filename = f"{date_str}{station}.MAG.csv"
        url = f"{MAG_BASE}{year}/{station}/{filename}"
        local_path = os.path.join(save_dir, filename)

        try:
            resp = requests.get(url, verify=False)
            with open(local_path, 'wb') as f:
                f.write(resp.content)
            downloaded.append(local_path)
            print(f"Downloaded {filename} to directory: {local_path}")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")

        current += timedelta(days=1)

    print(f"Done! {len(downloaded)} file(s) downloaded to directory: {save_dir}.")
    return downloaded

# Download three days of data for GILL station (e.g., January 1, 2008 to January 3, 2008)
mag_files = download_mag_batch("GILL", "20080101", "20080103")

#### 1.2 Manual Download

1. First visit the [CSA Open Data Portal - CARISMA Dataset (CSV)](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma_csv/mag/daily/)
2. Browse or search the files you need
3. Download and save them locally
4. Place the files into the /data/mag/ folder of this tutorial

### Part 2: Load and Explore Magnetometer Data

The magnetometer CSV files have a specific structure with comment lines, station metadata, and actual data. 

**What the files look like:**
```
#All materials produced using CARISMA data are subject to...
"#The authors thank I.R. Mann, D.K. Milling and the rest of the CARISMA team...
#The following CARISMA reference paper should also be cited: Mann I. R. et al. (2008)...
# Site Lat Long yyyymmdd CoordSys Units no of records
GILL  56.376 265.360 20080101 GEODETIC nT  1Hz
 
# Date(dd/mm/yyyy),time(hh:mi:ss),X,Y,Z,F=. if the data is valid anything else the data is suspect.
2008/01/01,00:00:00,10579.525,-717.959,59529.581,.
2008/01/01,00:00:01,10579.541,-717.950,59529.516,.
2008/01/01,00:00:02,10579.534,-717.947,59529.534,.
```

The key fields are:
- **X**: Geograpraphic northward component (nT)
- **Y**: Geographic eastward component (nT)
- **Z**: Vertical downward component (nT)
- **Flag**: `.` means valid data, anything else means the data is suspect


In [ ]:
def load_mag_data(file_path):
    data_lines = []
    metadata = {}

    with open(file_path, 'r') as f:
        for line in f:
            stripped = line.strip().strip('"')

            # Skip comment lines and blank lines
            if stripped.startswith("#") or stripped == "":
                continue
            
            # Detect station metadata line (no commans, space separated)
            if ',' not in stripped and len(stripped.split()) >= 6:
                parts = stripped.split()
                metadata= {
                    "station": parts[0],
                    "latitude": float(parts[1]),
                    "longitude": float(parts[2]),
                    "date": float(parts[3]),
                    "coord_system": parts[4],
                    "units": parts[5]
                }
                if len(metadata) > 6:
                    metadata['sampling'] = parts[6]
                continue

            # Otherwise, it's a data line
            data_lines.append(stripped)

    # Convert data lines to DataFrame
    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        names=["date", "time", "X", "Y", "Z", "flag"],
        header=None
    )

    # Combine date and time into a single datetime column
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])

    return df, metadata

In [ ]:
# Load one date of magnetometer data 
mag_df, mag_meta = load_mag_data("data/mag/20080101GILL.MAG.csv")

print("Station Metadata:")
for key, value in mag_meta.items():
    print(f"{key}: {value}")

print(f"\nDataframe shape: {mag_df.shape} (rows, columns)")
print(f"Time range: {mag_df['datetime'].min()} to {mag_df['datetime'].max()}")
print(f"Sampling interval: ~{mag_df['datetime'].diff().median().total_seconds():.0f} second(s)")

mag_df.head()

In [ ]:
# Explore the data
print("Magnetic field stats (nT):")
print(mag_df[["X", "Y", "Z"]].describe().round(2))

# Check data quality flags
valid_count = (mag_df['flag'] == '.').sum()
total_count = len(mag_df)
print(f"\nData Quality: {valid_count}/{total_count} records valid ({valid_count/total_count*100:.2f}%)")

### Part 3: Visualize Magnetometer Data

CARISMA magnetometers measure the X, Y, Z components of Earth's magnetic field in the geodetic coordinate system. 

| Component | Direction | Typical Range |
|-----------|-----------|---------------|
| **X** | Geographic North | ~10,000 nT |
| **Y** | Geographic East | ~-1,000 nT |
| **Z** | Vertical (positive downward) | ~59,000 nT |

In [ ]:
# Plot all three magnetic field components for one day 
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

components = ['X', 'Y', 'Z']
colors = ['tab:blue', 'tab:orange', 'tab:green']
labels = ['X (North)', 'Y (East)', 'Z (Down)']

for ax, comp, color, label in zip(axes, components, colors, labels):
    ax.plot(mag_df['datetime'], mag_df[comp], color=color, linewidth=0.5)
    ax.set_ylabel(f"{label}\n(nT)")
    ax.grid(True, alpha=0.3)

station_name = mag_meta.get("station", "Unknown")
date_label = mag_df['datetime'].dt.date.iloc[0]
axes[0].set_title(
    f"CARISMA Magnetometer Data for {station_name} on {date_label}",
    fontsize=14
)

axes[-1].set_xlabel("Time (UTC)")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=3))

plt.tight_layout()
plt.savefig("magnetometer_plot.png", dpi=150, bbox_inches='tight')
plt.show()

print("Saved: magnetometer_plot.png")

In [ ]:
# Zoomed in view: First 6 hours of the X component
cutoff = mag_df['datetime'].iloc[0] + pd.Timedelta(hours=6)
subset = mag_df[mag_df['datetime'] < cutoff]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(subset['datetime'], subset['X'], color='tab:blue', linewidth=0.7)
ax.set_label('Time (UTC)')
ax.set_ylabel("X (North) [nT]")
ax.set_title(f"Zoomed In: X Component for {station_name} on {date_label}", fontsize=14)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

In [ ]:
# Multi day plot: Combine all magnetometer files
mag_folder = "data/mag/"
all_mag_dfs = []

for f in sorted(os.listdir(mag_folder)):
    if f.endswith(".MAG.csv"):
        df, _ = load_mag_data(os.path.join(mag_folder, f))
        all_mag_dfs.append(df)

if len(all_mag_dfs) > 1:
    combined_df = pd.concat(all_mag_dfs, ignore_index=True)

    fig, ax = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

    for ax, comp, color, label in zip(ax, components, colors, labels):
        ax.plot(combined_df['datetime'], combined_df[comp], color=color, linewidth=0.3)
        ax.set_ylabel(f'{label}\n(nT)')
        ax.grid(True, alpha=0.3)

    axes[0].set_title(
        f"CARISMA Magnetometer Data for {station_name} (Multiple Days)",
        fontsize=16,
    )
    axes[-1].set_xlabel("Time (UTC)")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))

    plt.tight_layout()
    plt.show()